# Concurrencia y Manejo de Memoria en Ruby

Ruby ofrece varias herramientas para la ejecución concurrente y paralela. Es fundamental entender el **GVL (Global VM Lock)** y cómo Ruby gestiona los recursos para escribir aplicaciones escalables.

## 1. El GVL (Global VM Lock)

En MRI (Matz's Ruby Interpreter), el GVL es un mecanismo que impide que múltiples hilos ejecuten código Ruby simultáneamente en un mismo proceso. 

- **Hilos de Ruby**: Son hilos nativos del SO.
- **Concurrencia**: Sí hay (un hilo puede esperar I/O mientras otro ejecuta).
- **Paralelismo**: No hay para código puro Ruby, pero sí para extensiones en C o tareas de I/O.

## 2. Hilos (Threads)

Los hilos son la unidad básica de concurrencia en Ruby.

In [ ]:
threads = []

5.times do |i|
  threads << Thread.new do
    sleep(rand(0.1..0.5))
    puts "Hilo #{i} finalizado"
  end
end

threads.each(&:join)
puts "Todos los hilos terminaron"

## 3. Procesos (Forks)

Para lograr paralelismo real en sistemas multinúcleo, Ruby puede crear procesos hijos. Estos tienen su propio espacio de memoria (aunque usan *Copy-on-Write* para eficiencia).

In [ ]:
if Process.respond_to?(:fork)
  pid = fork do
    puts "Soy el proceso hijo con PID: #{Process.pid}"
    sleep 1
  end

  puts "Soy el proceso padre, esperando al hijo #{pid}..."
  Process.wait(pid)
else
  puts "Fork no soportado en esta plataforma (ej. Windows)"
end

## 4. Fibers (Fibras)

Las fibras son primitivas de concurrencia ligera (corrutinas) que ofrecen control manual sobre la pausa y reanudación del código.

In [ ]:
fibra = Fiber.new do
  puts "Hola desde la fibra"
  Fiber.yield "Dato de vuelta"
  puts "Fibra reanudada"
end

puts fibra.resume
fibra.resume

## 5. Ractors (Ruby 3.0+)

Los Ractors (Ruby Actors) permiten paralelismo real sin compartir memoria (o compartiendo memoria inmutable), eliminando las limitaciones del GVL para tareas CPU-bound.

In [ ]:
# Nota: Esta característica es experimental en algunas versiones
r = Ractor.new do
  # Los Ractors no pueden acceder a variables globales del exterior
  msg = Ractor.receive
  msg.upcase
end

r.send "mensaje paralelo"
puts "Resultado Ractor: #{r.take}"

## 6. Gestión de Memoria y GC

Ruby utiliza un Garbage Collector (GC) generacional y progresivo.

### Conceptos Clave:
- **ObjectSpace**: Donde viven todos los objetos.
- **Copy-on-Write (CoW)**: Los procesos hijos comparten memoria con el padre hasta que intentan escribir en ella.
- **GC Compacting**: Ruby puede compactar la memoria para reducir la fragmentación.

In [ ]:
require 'objspace'

# Estadísticas del GC
puts GC.stat

# Medir tamaño de un objeto
str = "x" * 1000
puts "Tamaño en bytes: #{ObjectSpace.memsize_of(str)}"

# Forzar el Garbage Collector
GC.start

## 7. Resumen de Herramientas

| Herramienta | Tipo | Paralelismo Real | Cuándo usar |
| :--- | :--- | :--- | :--- |
| **Threads** | Concurrente | No (en Ruby) | Tareas de I/O (Red, DB) |
| **Processes** | Paralelo | Sí | Tareas pesadas de CPU |
| **Fibers** | Cooperativo | No | Generadores o servidores asíncronos |
| **Ractors** | Paralelo | Sí | Paralelismo moderno seguro |